# 06-09 · Depthwise Separable Convolutions

**Learning objective:** Understand how depthwise separable convolutions factorize a standard convolution into a depth-wise and point-wise step, implement both MobileNet-V1 and V2 blocks from scratch, and quantify the parameter-efficiency vs accuracy trade-off on CIFAR-10.

**Prerequisites:** 06-05 (ResNet & Skip Connections), 06-08 (Batch Normalization).


## Part 0 — Setup & Prerequisites

A standard Conv2d with kernel size $k$, $C_{\text{in}}$ input channels, and $C_{\text{out}}$ output channels costs:

$$
k^2 \cdot C_{\text{in}} \cdot C_{\text{out}} \text{ parameters and MAdds per output pixel}
$$

**Depthwise Separable Convolution** (Howard et al. 2017, MobileNet) splits this into two cheaper operations:

1. **Depthwise conv** — one $k \times k$ filter *per input channel* (no cross-channel mixing):    $k^2 \cdot C_{\text{in}}$ parameters.
2. **Pointwise conv** — $1 \times 1$ conv to mix channels:    $C_{\text{in}} \cdot C_{\text{out}}$ parameters.

**Total:** $C_{\text{in}}(k^2 + C_{\text{out}})$ — a factor of $\frac{1}{C_{\text{out}}} + \frac{1}{k^2}$ fewer parameters than standard conv.  For $k=3$, $C_{\text{out}}=256$: **≈ 8–9× compression** with only minor accuracy loss.

This notebook implements both MobileNet-V1 (DWSep blocks) and MobileNet-V2 (Inverted Residual blocks) and compares them against a standard CNN on CIFAR-10.


In [ ]:
import sys, warnings, time, random, math
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import CIFAR10

warnings.filterwarnings("ignore")

from collections.abc import Callable
print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"NumPy  : {np.__version__}")
if torch.cuda.is_available():
    print(f"CUDA   : {torch.version.cuda}")
    print(f"GPU    : {torch.cuda.get_device_name(0)}")


In [ ]:
# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ── Hyperparameters ───────────────────────────────────────────────────────────
BATCH_SIZE    = 128
NUM_EPOCHS    = 8
LEARNING_RATE = 1e-3
NUM_CLASSES   = 10
SUBSET_SIZE   = 15_000
DATA_ROOT     = "data/"

# ── CIFAR-10 stats ────────────────────────────────────────────────────────────
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2023, 0.1994, 0.2010)
CLASSES      = ["airplane", "automobile", "bird", "cat", "deer",
                "dog", "frog", "horse", "ship", "truck"]

print(f"Training: BATCH_SIZE={BATCH_SIZE}  NUM_EPOCHS={NUM_EPOCHS}  SUBSET={SUBSET_SIZE:,}")


In [ ]:
train_tf = T.Compose([
    T.RandomHorizontalFlip(),
    T.RandomCrop(32, padding=4),
    T.ToTensor(),
    T.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])
plain_tf = T.Compose([
    T.ToTensor(),
    T.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

full_train  = CIFAR10(root=DATA_ROOT, train=True,  download=True, transform=train_tf)
plain_train = CIFAR10(root=DATA_ROOT, train=True,  download=True, transform=plain_tf)
test_set    = CIFAR10(root=DATA_ROOT, train=False, download=True, transform=plain_tf)

generator     = torch.Generator().manual_seed(SEED)
n_total       = len(full_train)
n_val         = int(0.1 * n_total)
n_train       = n_total - n_val
all_indices   = torch.randperm(n_total, generator=generator).tolist()
train_indices = all_indices[:n_train]
val_indices   = all_indices[n_train:]
sub_indices   = train_indices[:SUBSET_SIZE]

train_loader = DataLoader(
    Subset(full_train, sub_indices), batch_size=BATCH_SIZE, shuffle=True,
    num_workers=0, pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    Subset(plain_train, val_indices), batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_set, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=torch.cuda.is_available(),
)
print(f"Train subset: {SUBSET_SIZE:,}  |  Val: {n_val:,}  |  Test: {len(test_set):,}")


## Part 1 — Parameter Efficiency Analysis

Before implementing anything, let's compute the parameter counts analytically to build intuition for *why* depthwise separable convolutions are so efficient.

The **reduction ratio** of DWSep vs standard Conv for $k=3$ is:

$$
\text{ratio} = \frac{C_{\text{in}}(k^2 + C_{\text{out}})}{k^2 \cdot C_{\text{in}} \cdot C_{\text{out}}}
= \frac{1}{C_{\text{out}}} + \frac{1}{k^2}
$$

For a typical mid-network layer with $C_{\text{out}}=128$: ratio $\approx 0.008 + 0.111 = 0.119$ → **8.4× fewer parameters**.  Shallower networks with small $C_{\text{out}}$ see less benefit; very deep layers with large channel counts see the most.


In [ ]:
def standard_conv_params(
    in_channels: int,
    out_channels: int,
    kernel_size: int,
    bias: bool = False,
) -> int:
    '''Compute parameter count for a standard Conv2d.

    Args:
        in_channels: Number of input channels.
        out_channels: Number of output channels.
        kernel_size: Square kernel side length.
        bias: Whether to include a bias term.

    Returns:
        Integer parameter count.
    '''
    params = kernel_size ** 2 * in_channels * out_channels
    if bias:
        params += out_channels
    return params


def dw_sep_params(
    in_channels: int,
    out_channels: int,
    kernel_size: int,
    bias: bool = False,
) -> tuple[int, int, int]:
    '''Compute parameter count for depthwise separable convolution.

    Args:
        in_channels: Number of input channels.
        out_channels: Number of output channels.
        kernel_size: Depthwise kernel side length (pointwise is always 1x1).
        bias: Whether to include bias in both sub-layers.

    Returns:
        Tuple of (depthwise_params, pointwise_params, total_params).
    '''
    dw_params = kernel_size ** 2 * in_channels + (in_channels if bias else 0)
    pw_params = 1 * 1 * in_channels * out_channels + (out_channels if bias else 0)
    return dw_params, pw_params, dw_params + pw_params


def compute_reduction(in_channels: int, out_channels: int, kernel_size: int) -> float:
    '''Compute parameter reduction ratio: DWSep / Standard.

    Args:
        in_channels: Number of input channels.
        out_channels: Number of output channels.
        kernel_size: Convolution kernel size.

    Returns:
        Reduction ratio in (0, 1) — smaller means more efficient.
    '''
    std   = standard_conv_params(in_channels, out_channels, kernel_size)
    _, _, dws = dw_sep_params(in_channels, out_channels, kernel_size)
    return dws / std


# ── Print table across typical layer configurations ───────────────────────────
print("Parameter analysis: Standard Conv vs Depthwise Separable Conv (k=3)\n")
print(f"  {'Cin':>5}  {'Cout':>5}  {'Standard':>10}  {'DWSep':>8}  {'Ratio':>7}  {'Speedup':>8}")
print("─" * 55)

configs = [
    (3,   32),  (32,  64),  (64, 128), (128, 256),
    (256, 256), (256, 512), (512, 512),
]
for cin, cout in configs:
    std = standard_conv_params(cin, cout, 3)
    _, _, dws = dw_sep_params(cin, cout, 3)
    ratio   = dws / std
    speedup = std / dws
    print(f"  {cin:>5}  {cout:>5}  {std:>10,}  {dws:>8,}  {ratio:>7.3f}  {speedup:>7.1f}×")

print("\nTheoretical ratio: 1/Cout + 1/k²  (for k=3: 1/Cout + 0.111)")
print("Large Cout → closer to 1/9 = 11.1% of standard params → 9x compression.")

# ── Plot: reduction ratio vs Cout ─────────────────────────────────────────────
cout_range = np.arange(16, 513, 8)
ratio_curve = 1.0 / cout_range + 1.0 / (3**2)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(cout_range, ratio_curve * 100, lw=2, color="#2196F3")
ax.axhline(100.0 / 9, color="k", lw=1.5, linestyle="--",
           label=f"Theoretical limit: 1/k² = {100/9:.1f}% (k=3)")
ax.fill_between(cout_range, 0, ratio_curve * 100, alpha=0.15, color="#2196F3")
ax.set_xlabel("Output channels $C_{out}$", fontsize=11)
ax.set_ylabel("DWSep params / Standard params (%)", fontsize=11)
ax.set_title("Parameter Reduction Ratio vs $C_{out}$ (k=3)\n"
             "Deep layers with many channels benefit most",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Part 1b — Depthwise Convolution: From Loops to Groups

A **depthwise convolution** applies one filter to each input channel independently.  The key insight is that setting `groups=C_in` in `F.conv2d` exactly implements this: the $C_{\text{in}}$ input channels are split into $C_{\text{in}}$ groups of 1 channel each, and each group is convolved with exactly one output filter.

We first show the **loop-based manual implementation** for clarity, then demonstrate that the `groups` API gives identical results — confirming our understanding.


In [ ]:
def depthwise_conv_manual(
    x: torch.Tensor,
    weight: torch.Tensor,
    padding: int = 1,
) -> torch.Tensor:
    '''Apply depthwise convolution via an explicit channel loop (educational).

    Each input channel is convolved with its own single-channel kernel.
    This is equivalent to F.conv2d with groups=C_in but makes the
    per-channel independence explicit.

    Args:
        x: Input tensor of shape (N, C_in, H, W).
        weight: Filter tensor of shape (C_in, 1, k, k).
        padding: Zero-padding added to each spatial side.

    Returns:
        Output tensor of shape (N, C_in, H', W').
    '''
    n_channels = x.shape[1]
    out_channels_list: list[torch.Tensor] = []
    for c in range(n_channels):
        x_c = x[:, c : c + 1, :, :]          # (N, 1, H, W)
        w_c = weight[c : c + 1, :, :, :]     # (1, 1, k, k)
        out_channels_list.append(F.conv2d(x_c, w_c, padding=padding))
    return torch.cat(out_channels_list, dim=1)


def depthwise_conv_groups(
    x: torch.Tensor,
    weight: torch.Tensor,
    padding: int = 1,
) -> torch.Tensor:
    '''Apply depthwise convolution via F.conv2d groups parameter (efficient).

    Setting groups=C_in in F.conv2d divides C_in input channels into C_in
    groups of 1 channel, each convolved with one output filter — exactly
    a depthwise convolution.

    Args:
        x: Input tensor of shape (N, C_in, H, W).
        weight: Filter tensor of shape (C_in, 1, k, k).
        padding: Zero-padding added to each spatial side.

    Returns:
        Output tensor of shape (N, C_in, H', W').
    '''
    in_channels = x.shape[1]
    return F.conv2d(x, weight, padding=padding, groups=in_channels)


def pointwise_conv(
    x: torch.Tensor,
    weight: torch.Tensor,
) -> torch.Tensor:
    '''Apply pointwise (1x1) convolution to mix channels.

    A 1x1 convolution applies the same linear transform to every spatial
    position independently, mixing information across channels without
    spatial mixing.

    Args:
        x: Input tensor of shape (N, C_in, H, W).
        weight: Filter tensor of shape (C_out, C_in, 1, 1).

    Returns:
        Output tensor of shape (N, C_out, H, W).
    '''
    return F.conv2d(x, weight)


# ── Verify: manual loop == groups API ─────────────────────────────────────────
torch.manual_seed(SEED)
x_test  = torch.randn(4, 8, 16, 16)             # (N, C_in, H, W)
dw_weight = torch.randn(8, 1, 3, 3)             # (C_in, 1, k, k)

out_manual = depthwise_conv_manual(x_test, dw_weight, padding=1)
out_groups = depthwise_conv_groups(x_test, dw_weight, padding=1)

max_diff = (out_manual - out_groups).abs().max().item()
print(f"Depthwise conv: manual loop vs groups API")
print(f"  Input shape   : {tuple(x_test.shape)}")
print(f"  Weight shape  : {tuple(dw_weight.shape)}")
print(f"  Output shape  : {tuple(out_manual.shape)}")
print(f"  Max abs diff  : {max_diff:.2e}  {'✓' if max_diff < 1e-5 else '✗'}")

# ── Verify pointwise conv ──────────────────────────────────────────────────────
pw_weight = torch.randn(16, 8, 1, 1)           # (C_out, C_in, 1, 1)
out_pw    = pointwise_conv(x_test, pw_weight)
print(f"\nPointwise conv output shape: {tuple(out_pw.shape)}")
print(f"  Parameters: C_in×C_out = {8}×{16} = {8*16} (no k² factor)")


In [ ]:
class DepthwiseSeparableConv(nn.Module):
    '''Depthwise Separable Convolution: depthwise + pointwise + BN + ReLU.

    Factorises a standard Conv2d into:
      1. Depthwise: k×k conv with groups=C_in (spatial filtering per channel).
      2. Pointwise: 1×1 conv (cross-channel mixing).
    Each sub-layer is followed by BatchNorm + ReLU.

    Attributes:
        depthwise: Depthwise Conv2d (groups=in_channels).
        dw_bn: BatchNorm after depthwise.
        dw_act: ReLU after depthwise BN.
        pointwise: 1×1 Conv2d (channel mixing).
        pw_bn: BatchNorm after pointwise.
        pw_act: ReLU after pointwise BN.
    '''

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        stride: int = 1,
        padding: int = 1,
    ) -> None:
        '''Initialise DepthwiseSeparableConv.

        Args:
            in_channels: Number of input channels.
            out_channels: Number of output channels.
            stride: Stride for the depthwise conv (controls spatial downsampling).
            padding: Padding for the depthwise 3×3 conv.
        '''
        super().__init__()
        # Depthwise: one filter per input channel
        self.depthwise = nn.Conv2d(
            in_channels, in_channels, kernel_size=3,
            stride=stride, padding=padding,
            groups=in_channels, bias=False,
        )
        self.dw_bn  = nn.BatchNorm2d(in_channels)
        self.dw_act = nn.ReLU(inplace=True)

        # Pointwise: mix channels with 1×1 conv
        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.pw_bn  = nn.BatchNorm2d(out_channels)
        self.pw_act = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''Apply depthwise then pointwise convolution.

        Args:
            x: Input tensor (N, C_in, H, W).

        Returns:
            Output tensor (N, C_out, H', W').
        '''
        x = self.dw_act(self.dw_bn(self.depthwise(x)))
        x = self.pw_act(self.pw_bn(self.pointwise(x)))
        return x


# ── Shape + parameter check ────────────────────────────────────────────────────
_dws = DepthwiseSeparableConv(in_channels=64, out_channels=128)
_inp = torch.zeros(2, 64, 16, 16)
_out = _dws(_inp)
assert _out.shape == (2, 128, 16, 16), f"Unexpected shape: {_out.shape}"
n_dws = sum(p.numel() for p in _dws.parameters() if p.requires_grad)
n_std = standard_conv_params(64, 128, 3)
print(f"DepthwiseSeparableConv(64 → 128):")
print(f"  DWSep params: {n_dws:,}  |  Standard Conv params: {n_std:,}")
print(f"  Compression:  {n_std / n_dws:.1f}×  ({n_dws/n_std*100:.1f}% of standard)")
print(f"  Output shape: {tuple(_out.shape)}  ✓")
del _dws, _inp, _out


## Part 2 — MobileNet Architectures

### 2.1 MobileNet-V1 (Howard et al. 2017)

MobileNet-V1 replaces every standard Conv with a Depthwise Separable Conv block.  The full network is a stack of DWSep blocks with stride-2 downsampling at specific layers, finishing with global average pooling and a linear classifier.

**Width multiplier $\alpha$:** All channel counts are scaled by $\alpha \in (0, 1]$, trading accuracy for even fewer parameters.  For $\alpha = 1.0$ (full width), the network achieves ~70% top-1 on ImageNet; $\alpha = 0.25$ cuts params by 16× with moderate accuracy loss.

### 2.2 MobileNet-V2 (Sandler et al. 2018) — Inverted Residuals

MobileNet-V2 adds two key ideas over V1:

1. **Inverted Residual block:** Rather than compressing channels before the depthwise    conv (as in standard ResNet bottlenecks), V2 *expands* channels by a factor $t$    (typically 6) before the DW step, then projects back down.
2. **Linear bottleneck:** The final projection (PW) has **no activation** — applying    ReLU to a low-dimensional bottleneck destroys information.

$$
\text{Expand } 1\times1 \xrightarrow{t \cdot C} \text{Depthwise } 3\times3 \xrightarrow{} \text{Project } 1\times1
$$

A **residual** shortcut is added when stride=1 and $C_{\text{in}} = C_{\text{out}}$.


In [ ]:
def make_channel(base: int, width_mult: float) -> int:
    '''Scale channel count by width multiplier, rounded to nearest multiple of 8.

    Args:
        base: Base number of channels.
        width_mult: Multiplicative scaling factor in (0, 1].

    Returns:
        Scaled channel count, rounded up to nearest 8.
    '''
    scaled = base * width_mult
    return max(8, int(math.ceil(scaled / 8)) * 8)


class MobileNetV1CIFAR(nn.Module):
    '''MobileNet-V1 adapted for CIFAR-10 (32×32 input).

    Uses DepthwiseSeparableConv blocks with MaxPool for downsampling
    rather than stride-2 DW conv, to preserve spatial resolution on
    the small 32×32 images.

    Attributes:
        first_conv: Initial 3×3 Conv → BN → ReLU.
        blocks: Sequential stack of DepthwiseSeparableConv blocks.
        gap: Global average pooling.
        fc: Linear classifier head.
        width_mult: Width multiplier used to scale channels.
    '''

    def __init__(self, num_classes: int = 10, width_mult: float = 1.0) -> None:
        '''Initialise MobileNetV1CIFAR.

        Args:
            num_classes: Number of output classes.
            width_mult: Width multiplier α ∈ (0, 1] scaling all channel counts.
        '''
        super().__init__()
        self.width_mult = width_mult

        def ch(base: int) -> int:
            '''Scale base channel count by width_mult.

            Args:
                base: Base number of channels.

            Returns:
                Scaled channel count.
            '''
            return make_channel(base, width_mult)

        # (in_ch, out_ch, use_maxpool)
        layer_cfg: list[tuple[int, int, bool]] = [
            (ch(32),  ch(64),  False),
            (ch(64),  ch(128), True),   # ↓ 32→16
            (ch(128), ch(128), False),
            (ch(128), ch(256), True),   # ↓ 16→8
            (ch(256), ch(256), False),
            (ch(256), ch(512), True),   # ↓ 8→4
        ]

        self.first_conv = nn.Sequential(
            nn.Conv2d(3, ch(32), kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(ch(32)),
            nn.ReLU(inplace=True),
        )

        block_list: list[nn.Module] = []
        for in_ch, out_ch, use_pool in layer_cfg:
            block_list.append(DepthwiseSeparableConv(in_ch, out_ch))
            if use_pool:
                block_list.append(nn.MaxPool2d(2))
        self.blocks = nn.Sequential(*block_list)

        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc  = nn.Linear(ch(512), num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''Forward pass through MobileNet-V1.

        Args:
            x: Input image batch (N, 3, 32, 32).

        Returns:
            Logits tensor (N, num_classes).
        '''
        x = self.first_conv(x)
        x = self.blocks(x)
        x = self.gap(x).flatten(1)
        return self.fc(x)


# ── Parameter check ───────────────────────────────────────────────────────────
for wm in [0.25, 0.5, 0.75, 1.0]:
    m = MobileNetV1CIFAR(num_classes=NUM_CLASSES, width_mult=wm)
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"  MobileNetV1 α={wm:.2f}: {n:>8,} params")


### 2.3 Inverted Residual (V2 Block) in Detail

The inverted residual block has three stages:

| Stage | Operation | Purpose |
|-------|-----------|---------|
| **Expand** | $1\times 1$ conv: $C_{\text{in}} \to t \cdot C_{\text{in}}$ | Increase channels before DW — more channels = richer spatial mixing |
| **Depthwise** | $3\times 3$ DW conv (stride $s$) | Spatial filtering in high-dimensional space |
| **Project** | $1\times 1$ conv: $t \cdot C_{\text{in}} \to C_{\text{out}}$, **no ReLU** | Compress back; linear activation preserves information |

**Residual shortcut:** Added *only* when $s = 1$ and $C_{\text{in}} = C_{\text{out}}$ (adding a skip when channels differ would require a 1×1 proj, adding params for little gain).


In [ ]:
class InvertedResidual(nn.Module):
    '''Inverted Residual block from MobileNet-V2.

    Three stages: Expand (1×1 PW) → Depthwise (3×3 DW) → Project (1×1 PW, no ReLU).
    A residual shortcut is added when stride=1 and in_channels==out_channels.

    Attributes:
        use_residual: Whether to add a skip connection.
        expand_conv: 1×1 expansion conv (present only when t > 1).
        depthwise: 3×3 DW conv.
        project: 1×1 projection conv (no activation).
    '''

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        stride: int = 1,
        expand_ratio: int = 6,
    ) -> None:
        '''Initialise InvertedResidual.

        Args:
            in_channels: Number of input channels.
            out_channels: Number of output channels.
            stride: Depthwise conv stride (1 = same size, 2 = halve spatial dims).
            expand_ratio: Channel expansion factor t; hidden dim = in_channels * t.
        '''
        super().__init__()
        assert stride in (1, 2), f"Stride must be 1 or 2, got {stride}"
        self.use_residual = (stride == 1) and (in_channels == out_channels)

        hidden_dim = in_channels * expand_ratio

        layers: list[nn.Module] = []

        # Expand phase (skip if t == 1 to save one conv)
        if expand_ratio != 1:
            layers += [
                nn.Conv2d(in_channels, hidden_dim, kernel_size=1, bias=False),
                nn.BatchNorm2d(hidden_dim),
                nn.ReLU6(inplace=True),
            ]

        # Depthwise phase
        layers += [
            nn.Conv2d(hidden_dim, hidden_dim, kernel_size=3,
                      stride=stride, padding=1, groups=hidden_dim, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU6(inplace=True),
        ]

        # Project phase — NO activation (linear bottleneck)
        layers += [
            nn.Conv2d(hidden_dim, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
        ]

        self.expand_dw_project = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''Forward pass with optional residual shortcut.

        Args:
            x: Input tensor (N, in_channels, H, W).

        Returns:
            Output tensor (N, out_channels, H', W').
        '''
        out = self.expand_dw_project(x)
        if self.use_residual:
            out = out + x
        return out


class MobileNetV2CIFAR(nn.Module):
    '''MobileNet-V2 adapted for CIFAR-10 (32×32 input).

    Uses inverted residual blocks with configurable expand ratio.
    Downsampling via stride-2 in depthwise conv rather than MaxPool.

    Attributes:
        first_conv: Initial 3×3 Conv → BN → ReLU6.
        blocks: Stack of InvertedResidual blocks.
        last_conv: 1×1 Conv to expand to 256 channels before GAP.
        gap: Global average pooling.
        fc: Linear classifier.
    '''

    # (expand_ratio, out_channels, num_blocks, stride)
    BLOCK_CFG: list[tuple[int, int, int, int]] = [
        (1,  16, 1, 1),
        (6,  24, 2, 1),   # ↓ handled by first block stride
        (6,  32, 3, 2),
        (6,  64, 4, 2),
        (6,  96, 3, 1),
        (6, 160, 3, 2),
    ]

    def __init__(self, num_classes: int = 10) -> None:
        '''Initialise MobileNetV2CIFAR.

        Args:
            num_classes: Number of output classes.
        '''
        super().__init__()

        self.first_conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU6(inplace=True),
        )

        layers: list[nn.Module] = []
        in_ch = 32
        for expand_ratio, out_ch, n_blocks, stride in self.BLOCK_CFG:
            for block_idx in range(n_blocks):
                blk_stride = stride if block_idx == 0 else 1
                layers.append(InvertedResidual(in_ch, out_ch, blk_stride, expand_ratio))
                in_ch = out_ch
        self.blocks = nn.Sequential(*layers)

        self.last_conv = nn.Sequential(
            nn.Conv2d(in_ch, 256, kernel_size=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU6(inplace=True),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc  = nn.Linear(256, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''Forward pass through MobileNet-V2.

        Args:
            x: Input image batch (N, 3, 32, 32).

        Returns:
            Logits tensor (N, num_classes).
        '''
        x = self.first_conv(x)
        x = self.blocks(x)
        x = self.last_conv(x)
        x = self.gap(x).flatten(1)
        return self.fc(x)


# ── Parameter summary ─────────────────────────────────────────────────────────
_v2 = MobileNetV2CIFAR(NUM_CLASSES)
_v1 = MobileNetV1CIFAR(NUM_CLASSES, width_mult=1.0)
n_v1 = sum(p.numel() for p in _v1.parameters())
n_v2 = sum(p.numel() for p in _v2.parameters())
print(f"MobileNetV1CIFAR  params: {n_v1:>8,}")
print(f"MobileNetV2CIFAR  params: {n_v2:>8,}")
# Count inverted residuals that use skip connections
n_skip = sum(1 for m in _v2.modules() if isinstance(m, InvertedResidual) and m.use_residual)
n_total = sum(1 for m in _v2.modules() if isinstance(m, InvertedResidual))
print(f"V2 InvertedResidual blocks: {n_total} total, {n_skip} with residual shortcut")
del _v1, _v2


### 2.4 Inverted Residual: Internal Shape Flow

Tracing tensor shapes through each stage of an InvertedResidual block clarifies the 'inverted' nature: channels expand *before* the depthwise conv (from $C_{\text{in}}$ to $t \cdot C_{\text{in}}$), then compress *after* it (to $C_{\text{out}}$).  This is opposite to the standard ResNet bottleneck which compresses first.


In [ ]:
# ── Trace the internal tensor shapes through an InvertedResidual block ─────────
# Understanding how dimensions change at each stage helps build intuition for
# why "inverted" residuals are different from standard ResNet bottlenecks.

def trace_inverted_residual(
    in_channels: int,
    out_channels: int,
    expand_ratio: int,
    stride: int = 1,
    input_hw: int = 8,
) -> list[tuple[str, tuple[int, ...]]]:
    '''Trace tensor shapes through an InvertedResidual block.

    Args:
        in_channels: Block input channel count.
        out_channels: Block output channel count.
        expand_ratio: Expansion factor t.
        stride: Depthwise conv stride.
        input_hw: Spatial size of the input feature map (assumed square).

    Returns:
        List of (stage_name, output_shape) tuples.
    '''
    block = InvertedResidual(in_channels, out_channels, stride, expand_ratio)
    block.eval()

    shapes: list[tuple[str, tuple[int, ...]]] = []
    x = torch.zeros(1, in_channels, input_hw, input_hw)
    shapes.append(("Input", tuple(x.shape)))

    # Walk through layers manually and record intermediate shapes
    with torch.no_grad():
        for idx, layer in enumerate(block.expand_dw_project.children()):
            x = layer(x)
            layer_type = type(layer).__name__
            shapes.append((f"Layer {idx}: {layer_type}", tuple(x.shape)))

    return shapes


# ── Example: in=16, out=24, t=6, stride=2 ─────────────────────────────────────
print("InvertedResidual shape flow  (in=16, out=24, t=6, stride=2):")
print("(This block does NOT add residual: stride=2 and in≠out)\n")
flow = trace_inverted_residual(16, 24, expand_ratio=6, stride=2, input_hw=16)
for stage_name, shape in flow:
    print(f"  {stage_name:<30}  → {shape}")

print("\nInvertedResidual shape flow  (in=24, out=24, t=6, stride=1):")
print("(This block DOES add residual: stride=1 and in==out)\n")
flow_skip = trace_inverted_residual(24, 24, expand_ratio=6, stride=1, input_hw=8)
for stage_name, shape in flow_skip:
    print(f"  {stage_name:<30}  → {shape}")

# ── Parameter breakdown per stage ─────────────────────────────────────────────
def inverted_residual_params_breakdown(
    in_channels: int,
    out_channels: int,
    expand_ratio: int,
) -> dict[str, int]:
    '''Compute per-stage parameter counts for an InvertedResidual block.

    Args:
        in_channels: Input channel count.
        out_channels: Output channel count.
        expand_ratio: Channel expansion factor t.

    Returns:
        Dict with keys "expand", "depthwise", "project", "total".
    '''
    hidden = in_channels * expand_ratio
    expand_p   = in_channels * hidden if expand_ratio > 1 else 0
    dw_p       = 3 * 3 * hidden
    project_p  = hidden * out_channels
    return {
        "expand":   expand_p,
        "depthwise": dw_p,
        "project":  project_p,
        "total":    expand_p + dw_p + project_p,
    }


print("\nPer-stage parameter breakdown (in=64, out=96, t=6):")
breakdown = inverted_residual_params_breakdown(64, 96, expand_ratio=6)
for stage, count in breakdown.items():
    pct = count / breakdown["total"] * 100 if breakdown["total"] > 0 else 0
    print(f"  {stage:<12}: {count:>6,} params  ({pct:>5.1f}%)")
print("  Pointwise projection dominates — not the DW conv.")


## Part 3 — Training & Comparison

We train three models on the same 15,000-sample CIFAR-10 subset:

| Model | Convolution type | Architecture |
|-------|-----------------|--------------|
| **StandardCNN** | Regular Conv2d + BN | Baseline — no factorisation |
| **MobileNetV1** | DWSep blocks (α=1.0) | Full-width V1 |
| **MobileNetV2** | Inverted Residual blocks | V2 with t=6 expansion |

All models use the same optimizer (Adam, lr=1e-3), epochs (8), and data split.


In [ ]:
class ConvBNReLU(nn.Module):
    '''Standard Conv2d → BN → ReLU building block.

    Attributes:
        block: Sequential container (Conv → BN → ReLU).
    '''

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
    ) -> None:
        '''Initialise ConvBNReLU.

        Args:
            in_channels: Number of input channels.
            out_channels: Number of output channels.
            kernel_size: Convolution kernel size.
            stride: Convolution stride.
            padding: Convolution zero-padding.
        '''
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size,
                      stride=stride, padding=padding, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''Apply Conv → BN → ReLU.

        Args:
            x: Input tensor (N, in_channels, H, W).

        Returns:
            Output tensor (N, out_channels, H', W').
        '''
        return self.block(x)


class StandardCNN(nn.Module):
    '''Standard CNN with regular Conv2d blocks — parameter-rich baseline.

    Attributes:
        features: Six ConvBNReLU + three MaxPool layers.
        gap: Global average pooling.
        fc: Linear classifier.
    '''

    def __init__(self, num_classes: int = 10) -> None:
        '''Initialise StandardCNN.

        Args:
            num_classes: Number of output classes.
        '''
        super().__init__()
        self.features = nn.Sequential(
            ConvBNReLU(3,   32),
            ConvBNReLU(32,  64),
            nn.MaxPool2d(2),         # 32→16
            ConvBNReLU(64,  128),
            ConvBNReLU(128, 256),
            nn.MaxPool2d(2),         # 16→8
            ConvBNReLU(256, 512),
            nn.MaxPool2d(2),         # 8→4
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc  = nn.Linear(512, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''Forward: features → GAP → linear.

        Args:
            x: Input image batch (N, 3, 32, 32).

        Returns:
            Logits tensor (N, num_classes).
        '''
        x = self.features(x)
        x = self.gap(x).flatten(1)
        return self.fc(x)


# ── Check parameter counts ────────────────────────────────────────────────────
for name, model in [
    ("StandardCNN",   StandardCNN(NUM_CLASSES)),
    ("MobileNetV1",   MobileNetV1CIFAR(NUM_CLASSES)),
    ("MobileNetV2",   MobileNetV2CIFAR(NUM_CLASSES)),
]:
    n_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  {name:<16}: {n_p:>8,} params")


In [ ]:
def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float]:
    '''Train for one epoch and return loss and accuracy.

    Args:
        model: Neural network to train.
        dataloader: Training DataLoader.
        optimizer: Optimiser instance.
        criterion: Loss function.
        device: Compute device.

    Returns:
        Tuple of (average_loss, accuracy).
    '''
    model.train()
    running_loss = 0.0
    correct      = 0
    total        = 0

    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct      += logits.argmax(1).eq(labels).sum().item()
        total        += labels.size(0)

    return running_loss / total, correct / total


def evaluate(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float]:
    '''Evaluate model on a dataset.

    Args:
        model: Neural network to evaluate.
        dataloader: Validation or test DataLoader.
        criterion: Loss function.
        device: Compute device.

    Returns:
        Tuple of (average_loss, accuracy).
    '''
    model.eval()
    running_loss = 0.0
    correct      = 0
    total        = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            logits          = model(images)
            loss            = criterion(logits, labels)
            running_loss   += loss.item() * images.size(0)
            correct        += logits.argmax(1).eq(labels).sum().item()
            total          += labels.size(0)

    return running_loss / total, correct / total


print("Training utilities defined: train_one_epoch(), evaluate()")


In [ ]:
model_configs: list[tuple[str, nn.Module]] = [
    ("StandardCNN",  StandardCNN(NUM_CLASSES)),
    ("MobileNetV1",  MobileNetV1CIFAR(NUM_CLASSES, width_mult=1.0)),
    ("MobileNetV2",  MobileNetV2CIFAR(NUM_CLASSES)),
]

histories: dict[str, dict] = {}
criterion = nn.CrossEntropyLoss()

for model_name, model in model_configs:
    print(f"\n{'='*56}")
    print(f"  Training: {model_name}")
    print(f"{'='*56}")

    torch.manual_seed(SEED)
    random.seed(SEED)
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    history: dict[str, list] = {
        "train_losses": [], "val_losses": [],
        "train_accs":   [], "val_accs":   [],
    }
    best_val_loss    = float("inf")
    best_model_state: dict = {}

    for epoch in range(NUM_EPOCHS):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss,   val_acc   = evaluate(model, val_loader, criterion, device)
        elapsed = time.time() - t0

        history["train_losses"].append(train_loss)
        history["val_losses"].append(val_loss)
        history["train_accs"].append(train_acc)
        history["val_accs"].append(val_acc)

        if val_loss < best_val_loss:
            best_val_loss    = val_loss
            best_model_state = {k: v.clone() for k, v in model.state_dict().items()}

        print(f"  Epoch {epoch+1}/{NUM_EPOCHS} | "
              f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
              f"Val Acc: {val_acc:.2%} | Time: {elapsed:.1f}s")

    model.load_state_dict(best_model_state)
    history["model"]     = model
    history["n_params"]  = sum(p.numel() for p in model.parameters() if p.requires_grad)
    histories[model_name] = history

print("\nAll models trained.")


In [ ]:
epochs_range = list(range(1, NUM_EPOCHS + 1))
model_colors = {
    "StandardCNN":  "#9E9E9E",
    "MobileNetV1":  "#2196F3",
    "MobileNetV2":  "#4CAF50",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for model_name, history in histories.items():
    col = model_colors[model_name]
    axes[0].plot(epochs_range, history["val_losses"],
                 lw=2, color=col, label=model_name, marker="o", markersize=4)
    axes[1].plot(epochs_range, [v * 100 for v in history["val_accs"]],
                 lw=2, color=col, label=model_name, marker="o", markersize=4)

for ax, ylabel, title in [
    (axes[0], "Val Loss",         "Validation Loss"),
    (axes[1], "Val Accuracy (%)", "Validation Accuracy"),
]:
    ax.set_xlabel("Epoch", fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle("StandardCNN vs MobileNet-V1 vs MobileNet-V2 on CIFAR-10",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


## Part 4 — Evaluation & Analysis

### 4.1 Parameter Efficiency: Accuracy per Million Parameters

Raw accuracy comparisons favour large models.  A more useful metric is **accuracy per million parameters** — how much accuracy does each model buy per unit of complexity?  Efficient models (MobileNet family) should score higher than brute-force over-parameterised baselines.


In [ ]:
# ── Full evaluation + efficiency table ───────────────────────────────────────
rows = []
for model_name, history in histories.items():
    model   = history["model"]
    n_params = history["n_params"]
    _, val_acc  = evaluate(model, val_loader,  criterion, device)
    _, test_acc = evaluate(model, test_loader, criterion, device)

    acc_per_million = test_acc * 100 / (n_params / 1e6)

    rows.append({
        "Model":         model_name,
        "Params (M)":    round(n_params / 1e6, 3),
        "Val Acc (%)":   round(val_acc  * 100, 2),
        "Test Acc (%)":  round(test_acc * 100, 2),
        "Acc/M params":  round(acc_per_million, 1),
    })

results_df = pd.DataFrame(rows).sort_values("Test Acc (%)", ascending=False)
results_df = results_df.reset_index(drop=True)
print(results_df.to_string(index=False))

# ── Bar chart: acc/million params ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
names  = results_df["Model"].tolist()
colors = [model_colors[n] for n in names]

axes[0].bar(names, results_df["Test Acc (%)"].tolist(), color=colors, edgecolor="k", lw=0.7)
axes[0].set_ylabel("Test Accuracy (%)", fontsize=11)
axes[0].set_title("Absolute Test Accuracy", fontsize=11, fontweight="bold")
axes[0].grid(axis="y", alpha=0.3)
for bar_v, val_v in zip(axes[0].patches, results_df["Test Acc (%)"].tolist()):
    axes[0].text(bar_v.get_x() + bar_v.get_width() / 2, bar_v.get_height() + 0.1,
                 f"{val_v:.1f}%", ha="center", fontsize=9)

axes[1].bar(names, results_df["Acc/M params"].tolist(), color=colors, edgecolor="k", lw=0.7)
axes[1].set_ylabel("Test Acc (%) / M params", fontsize=11)
axes[1].set_title("Parameter Efficiency", fontsize=11, fontweight="bold")
axes[1].grid(axis="y", alpha=0.3)
for bar_v, val_v in zip(axes[1].patches, results_df["Acc/M params"].tolist()):
    axes[1].text(bar_v.get_x() + bar_v.get_width() / 2, bar_v.get_height() + 0.1,
                 f"{val_v:.1f}", ha="center", fontsize=9)

plt.suptitle("Accuracy vs Parameter Efficiency", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# ── Approximate FLOPs for a single forward pass ───────────────────────────────
# We count multiply-add operations (MAdds) analytically.
# For Conv2d: MAdds = k² × C_in × C_out × H_out × W_out
# For DWSep:  MAdds = k² × C_in × H × W  (DW)  +  C_in × C_out × H × W  (PW)

def count_conv_madds(
    in_channels: int,
    out_channels: int,
    kernel_size: int,
    h_out: int,
    w_out: int,
    groups: int = 1,
) -> int:
    '''Count multiply-add operations for one Conv2d layer.

    Args:
        in_channels: Number of input channels.
        out_channels: Number of output channels.
        kernel_size: Square kernel side length.
        h_out: Output feature map height.
        w_out: Output feature map width.
        groups: Number of Conv groups (1 = standard, C_in = depthwise).

    Returns:
        Number of multiply-add operations.
    '''
    ch_per_group = in_channels // groups
    return kernel_size ** 2 * ch_per_group * out_channels * h_out * w_out


def estimate_madds(
    model: nn.Module,
    input_shape: tuple[int, int, int, int],
) -> int:
    '''Estimate total MAdds by hooking into Conv2d layers.

    Registers forward hooks that record output shapes, then computes MAdds
    analytically for every Conv2d in the model.

    Args:
        model: PyTorch model.
        input_shape: (N, C, H, W) input tensor shape.

    Returns:
        Total approximate MAdds (integer).
    '''
    total_madds = 0
    hooks: list = []

    def make_hook(in_ch: int, out_ch: int, k: int, grps: int) -> "Callable":
        '''Create a forward hook to count MAdds for one Conv layer.

        Args:
            in_ch: Input channels.
            out_ch: Output channels.
            k: Kernel size.
            grps: Number of groups.

        Returns:
            Hook function that updates total_madds.
        '''

        def hook(
            module: nn.Module,
            inp: tuple,
            out: torch.Tensor,
        ) -> None:
            '''Record MAdds from output spatial dimensions.

            Args:
                module: Conv layer.
                inp: Unused inputs.
                out: Output tensor.
            '''
            nonlocal total_madds
            _, _, h_out, w_out = out.shape
            total_madds += count_conv_madds(in_ch, out_ch, k, h_out, w_out, grps)
        return hook

    for module in model.modules():
        if isinstance(module, nn.Conv2d):
            h = make_hook(
                module.in_channels, module.out_channels,
                module.kernel_size[0], module.groups,
            )
            hooks.append(module.register_forward_hook(h))

    dummy = torch.zeros(*input_shape)
    model.eval()
    with torch.no_grad():
        _ = model(dummy)

    for hook in hooks:
        hook.remove()

    return total_madds


print("Estimated MAdds for a single 32×32 image:\n")
print(f"  {'Model':<16}  {'Params (M)':>11}  {'MAdds (M)':>10}  {'MAdds/Param':>12}")
print("─" * 55)

input_shape = (1, 3, 32, 32)
for model_name, history in histories.items():
    model   = history["model"]
    n_p     = history["n_params"]
    n_madds = estimate_madds(model, input_shape)
    ratio   = n_madds / n_p if n_p > 0 else 0.0
    print(f"  {model_name:<16}  {n_p/1e6:>11.3f}  {n_madds/1e6:>10.2f}  {ratio:>12.2f}")

print("\nMobileNets trade parameters for MAdds differently —")
print("V2's inverted residuals increase MAdds (expanded channels) but improve accuracy.")


### 4.2 Inference Latency Benchmark

FLOPs are a theoretical estimate; actual inference latency depends on memory access patterns, cache behaviour, and hardware parallelism.  We benchmark all three models directly on CPU to measure real-world throughput at different batch sizes.


In [ ]:
# ── CPU inference latency benchmark ───────────────────────────────────────────
# We time forward passes at different batch sizes to compare latency and
# throughput. MobileNets are designed for inference efficiency, so they
# should show better throughput per parameter than StandardCNN.

N_WARMUP    = 5     # warmup iterations (not timed)
N_TIMING    = 50    # timed iterations
BATCH_SIZES = [1, 4, 16, 32, 64]


def benchmark_inference(
    model: nn.Module,
    batch_size: int,
    input_shape: tuple[int, int, int],
    device: torch.device,
    n_warmup: int = 5,
    n_timing: int = 50,
) -> dict[str, float]:
    '''Measure inference latency and throughput for a single model.

    Args:
        model: Neural network to benchmark.
        batch_size: Number of images per forward pass.
        input_shape: (C, H, W) input shape per image.
        device: Compute device.
        n_warmup: Number of warmup passes (not included in timing).
        n_timing: Number of timed forward passes.

    Returns:
        Dict with "latency_ms" (per batch) and "throughput_img_s" (images/sec).
    '''
    model.eval()
    dummy = torch.zeros(batch_size, *input_shape, device=device)

    # Warmup
    with torch.no_grad():
        for _ in range(n_warmup):
            _ = model(dummy)

    # Timed runs
    t_start = time.perf_counter()
    with torch.no_grad():
        for _ in range(n_timing):
            _ = model(dummy)
    t_total = time.perf_counter() - t_start

    latency_ms  = t_total / n_timing * 1000
    throughput  = batch_size * n_timing / t_total

    return {"latency_ms": latency_ms, "throughput_img_s": throughput}


print(f"Inference latency benchmark (CPU, {N_TIMING} timed passes):\n")
latency_rows = []

for batch_size in BATCH_SIZES:
    print(f"  Batch size = {batch_size}")
    for model_name, history in histories.items():
        model = history["model"]
        result = benchmark_inference(
            model, batch_size, (3, 32, 32), torch.device("cpu"),
            n_warmup=N_WARMUP, n_timing=N_TIMING,
        )
        n_p = history["n_params"]
        latency_rows.append({
            "Model":           model_name,
            "Batch size":      batch_size,
            "Latency (ms)":    round(result["latency_ms"], 2),
            "Throughput (i/s)": round(result["throughput_img_s"], 0),
            "Params (M)":      round(n_p / 1e6, 3),
        })
        print(f"    {model_name:<16}: {result['latency_ms']:>7.2f} ms/batch  "
              f"| {result['throughput_img_s']:>7.0f} img/s")
    print()

# ── Plot: throughput vs batch size ─────────────────────────────────────────────
latency_df = pd.DataFrame(latency_rows)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for model_name, grp in latency_df.groupby("Model"):
    col = model_colors[model_name]
    axes[0].plot(grp["Batch size"], grp["Latency (ms)"],
                 marker="o", lw=2, color=col, label=model_name, markersize=5)
    axes[1].plot(grp["Batch size"], grp["Throughput (i/s)"],
                 marker="o", lw=2, color=col, label=model_name, markersize=5)

for ax, ylabel, title in [
    (axes[0], "Latency (ms / batch)", "Batch Latency"),
    (axes[1], "Throughput (images / sec)", "Inference Throughput"),
]:
    ax.set_xlabel("Batch size", fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    ax.set_xticks(BATCH_SIZES)

plt.suptitle("CPU Inference Latency & Throughput", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()
print("MobileNets achieve higher throughput per parameter due to fewer FLOPs.")


In [ ]:
# ── Visualise learned depthwise filters from MobileNetV1 ─────────────────────
# Depthwise filters are (C_in, 1, k, k) — one 3×3 filter per input channel.
# Unlike standard conv filters (C_out, C_in, 3, 3), DW filters are scalar-valued
# per channel and visualise the spatial patterns each channel responds to.

def get_first_dw_filters(
    model: nn.Module,
) -> torch.Tensor | None:
    '''Extract the depthwise filter weights from the first DWSep block.

    Args:
        model: MobileNetV1CIFAR model.

    Returns:
        Depthwise filter tensor of shape (C_in, 1, k, k) or None if not found.
    '''
    for module in model.modules():
        if isinstance(module, DepthwiseSeparableConv):
            return module.depthwise.weight.detach().cpu()
    return None


v1_model     = histories["MobileNetV1"]["model"]
dw_filters   = get_first_dw_filters(v1_model)    # (C_in, 1, 3, 3)

if dw_filters is not None:
    n_filters = dw_filters.shape[0]
    n_show    = min(32, n_filters)
    n_cols    = 8
    n_rows    = math.ceil(n_show / n_cols)

    # Normalise each filter independently for display
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, n_rows * 1.5))
    axes_flat = axes.flatten() if n_rows > 1 else axes

    for idx in range(n_show):
        filt = dw_filters[idx, 0].numpy()   # (3, 3)
        # Normalise to [0, 1] for display
        filt_min, filt_max = filt.min(), filt.max()
        filt_norm = (filt - filt_min) / max(filt_max - filt_min, 1e-8)
        axes_flat[idx].imshow(filt_norm, cmap="RdBu_r", vmin=0, vmax=1)
        axes_flat[idx].axis("off")
        axes_flat[idx].set_title(f"Ch {idx}", fontsize=6)

    for idx in range(n_show, len(axes_flat)):
        axes_flat[idx].axis("off")

    fig.suptitle(f"Learned Depthwise Filters — MobileNetV1, First DWSep Block\n"
                 f"(First {n_show} of {n_filters} channels; each 3×3 spatial filter)",
                 fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.show()
    print(f"Depthwise filter shape: {tuple(dw_filters.shape)}  (C_in, 1, k, k)")
    print("Each filter captures a different spatial pattern: edges, blobs, diagonal gradients.")


### 4.4 Per-Class Accuracy Breakdown

Aggregate accuracy hides class-level variation.  The heatmap below shows which CIFAR-10 classes each model handles well and where each architecture struggles — which often reveals structural biases (e.g., texture-sensitive vs shape-sensitive features).


In [ ]:
# ── Per-class accuracy breakdown for all three models ────────────────────────
def per_class_accuracy(
    model: nn.Module,
    dataloader: DataLoader,
    device: torch.device,
    num_classes: int = 10,
) -> np.ndarray:
    '''Compute per-class top-1 accuracy.

    Args:
        model: Trained model in eval mode.
        dataloader: DataLoader for evaluation.
        device: Compute device.
        num_classes: Number of output classes.

    Returns:
        Float array of shape (num_classes,) with per-class accuracy in [0,1].
    '''
    model.eval()
    class_correct = torch.zeros(num_classes)
    class_total   = torch.zeros(num_classes)

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            preds = model(images).argmax(1)
            for cls_idx in range(num_classes):
                mask = labels == cls_idx
                if mask.sum() == 0:
                    continue
                class_correct[cls_idx] += preds[mask].eq(labels[mask]).sum().item()
                class_total[cls_idx]   += mask.sum().item()

    return (class_correct / class_total.clamp(min=1)).numpy()


all_class_accs: dict[str, np.ndarray] = {}
for model_name, history in histories.items():
    all_class_accs[model_name] = per_class_accuracy(
        history["model"], val_loader, device, NUM_CLASSES
    )

model_list = list(histories.keys())
acc_matrix = np.array([all_class_accs[name] for name in model_list]) * 100

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(acc_matrix, vmin=0, vmax=100, cmap="YlGnBu", aspect="auto")
ax.set_xticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASSES, rotation=45, ha="right", fontsize=9)
ax.set_yticks(range(len(model_list)))
ax.set_yticklabels(model_list, fontsize=9)

for row_idx in range(len(model_list)):
    for col_idx in range(NUM_CLASSES):
        val_v   = acc_matrix[row_idx, col_idx]
        text_c  = "white" if val_v < 40 else "black"
        ax.text(col_idx, row_idx, f"{val_v:.0f}",
                ha="center", va="center", fontsize=9, color=text_c)

plt.colorbar(im, ax=ax, label="Val Accuracy (%)")
ax.set_title("Per-Class Val Accuracy (%) — Model × CIFAR-10 Class",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

# ── Classes with biggest accuracy difference between StandardCNN and MobileNetV2 ──
print("\nClasses where MobileNetV2 outperforms StandardCNN most:")
diff = all_class_accs["MobileNetV2"] - all_class_accs["StandardCNN"]
sorted_classes = sorted(enumerate(diff), key=lambda t: t[1], reverse=True)
for cls_idx, gain in sorted_classes[:5]:
    print(f"  {CLASSES[cls_idx]:<12}: {gain:>+.1%}")


### 4.3 Width Multiplier Ablation

The width multiplier $\alpha$ in MobileNet-V1 scales all channel counts by a constant factor, enabling a smooth accuracy-vs-parameter trade-off curve.  This is useful when deploying to devices with different compute budgets.

We train V1 at four width multipliers (0.25, 0.5, 0.75, 1.0) for 5 epochs each and plot the resulting accuracy-vs-parameter Pareto curve.


In [ ]:
WIDTH_MULT_EPOCHS = 5
width_mults = [0.25, 0.5, 0.75, 1.0]
width_results: list[dict] = []

print(f"Width multiplier sweep ({WIDTH_MULT_EPOCHS} epochs each):\n")
print(f"  {'α':>5}  {'Params':>8}  {'Val Acc':>8}  {'Acc/M params':>13}")
print("─" * 42)

for wm in width_mults:
    torch.manual_seed(SEED)
    random.seed(SEED)

    wm_model = MobileNetV1CIFAR(NUM_CLASSES, width_mult=wm).to(device)
    n_wm     = sum(p.numel() for p in wm_model.parameters())
    wm_opt   = torch.optim.Adam(wm_model.parameters(), lr=LEARNING_RATE)
    wm_crit  = nn.CrossEntropyLoss()

    best_va   = 0.0
    best_state: dict = {}

    for epoch in range(WIDTH_MULT_EPOCHS):
        train_one_epoch(wm_model, train_loader, wm_opt, wm_crit, device)
        _, va = evaluate(wm_model, val_loader, wm_crit, device)
        if va > best_va:
            best_va   = va
            best_state = {k: v.clone() for k, v in wm_model.state_dict().items()}

    width_results.append({
        "alpha":    wm,
        "n_params": n_wm,
        "val_acc":  best_va,
    })
    eff = best_va * 100 / (n_wm / 1e6)
    print(f"  {wm:>5.2f}  {n_wm:>8,}  {best_va:>8.2%}  {eff:>13.1f}")

# ── Accuracy-parameter Pareto curve ───────────────────────────────────────────
params_m = [r["n_params"] / 1e6 for r in width_results]
accs_pct = [r["val_acc"] * 100 for r in width_results]
alphas   = [r["alpha"] for r in width_results]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(params_m, accs_pct, marker="o", lw=2, color="#2196F3", markersize=9)
for x_pt, y_pt, alpha_v in zip(params_m, accs_pct, alphas):
    ax.annotate(f"α={alpha_v}", (x_pt, y_pt), textcoords="offset points",
                xytext=(8, 4), fontsize=10)

ax.set_xlabel("Parameters (Millions)", fontsize=11)
ax.set_ylabel("Val Accuracy (%)", fontsize=11)
ax.set_title("MobileNetV1 Width Multiplier Pareto Curve\n"
             "Accuracy vs Parameter Count",
             fontsize=11, fontweight="bold")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\nEach point on the curve represents a different width multiplier.")
print("Choose α based on the device compute budget: α=0.25 for microcontrollers,")
print("α=1.0 for mobile phones, and beyond (no width reduction) for cloud inference.")


## Part 5 — Summary & Lessons Learned

### Key Takeaways

- **Depthwise convolution** applies one $k \times k$ filter *per channel*   (implemented via `groups=C_in` in `F.conv2d`), separating spatial filtering   from cross-channel mixing.
- **Depthwise Separable Convolution** (DW + pointwise) achieves   $\frac{1}{C_{\text{out}}} + \frac{1}{k^2}$ fewer parameters than standard Conv —   roughly **8–9× fewer** for typical mid-network layers.
- **MobileNet-V1** stacks DWSep blocks with optional stride-2 depthwise for   downsampling; the **width multiplier** $\alpha$ scales all channels uniformly,   yielding a smooth accuracy-vs-parameter trade-off curve.
- **MobileNet-V2** introduces **Inverted Residual blocks**: expand channels →   depthwise (in high-dim space) → project (no activation, linear bottleneck).   Residual shortcuts are added when stride=1 and channel dims match.
- **Parameter efficiency** (accuracy per million params) often favours MobileNet   architectures over brute-force larger models — especially on constrained hardware.

### What's Next

→ **06-10 (CNN Pipeline Capstone)** synthesises all Module 06 techniques — architecture search, data augmentation, BN, depthwise conv — into an end-to-end CIFAR-10 pipeline with ONNX export.
